# Session reports

> Build session-level report data from monitoring-session metadata and foreground-event records.

This module converts session and foreground-event data into a nested dictionary for the reporting UI. It parses timestamps, calculates non-negative whole-second event durations, derives display names from window titles, and aggregates activity by application and title.

Idle intervals contribute to session summary metrics but are excluded from application and title rankings. Empty event input produces a zeroed summary and an empty event breakdown. The module performs no database access or UI rendering.


In [ ]:
#| default_exp reporter

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import pandas as pd

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import snooper_pkg.config as cf

## Report construction


In [ ]:
#| export
def build_session_report_data(session, events):
    "Return session metadata, summary metrics, and event breakdowns as a report dictionary."
    report_data_dict = {}
    report_data_dict['session'] = _session_data(session)
    events_df = _clean_events(events)
    report_data_dict['summary'] = _summary_data(events_df)
    report_data_dict['events'] = _events_data(events_df)
    return report_data_dict

## Session and event preparation


In [ ]:
#| export
from datetime import datetime

def _session_data(session):
    return {'start_time': session['start_time'],
            'end_time': session['end_time'],
            'duration_seconds': ((datetime.fromisoformat(session['end_time']) - datetime.fromisoformat(session['start_time']))
            .total_seconds())}

In [ ]:
#| export
def _clean_events(events):    
    df = pd.DataFrame(events).copy()
    if len(df) == 0: return df
    df['start_time'] = pd.to_datetime(df['start_time'])
    df['end_time'] = pd.to_datetime(df['end_time'])
    df['duration_seconds'] = (
        df['end_time'] - df['start_time']
    ).dt.total_seconds().clip(lower=0).astype(int)

    df['app_name'] = df['window_title'].apply(lambda x: x.split(' - ')[-1])
    df['clean_title'] = df['window_title'].apply(lambda x: x.split(' - ')[-2] if ' - 'in x else 'Unknown')
    return df

## Event aggregation

In [ ]:
#| export
def _events_data(events_df, top_n_apps = cf.NUM_TOP_APP_IN_REPORT , top_n_titles=cf.NUM_TOP_WINDOW_IN_REPORT):
    if len(events_df) == 0:
        return []

    active_df = events_df[events_df['is_idle'] == False].copy()
    if active_df.empty:
        return []

    total_active_seconds = int(active_df['duration_seconds'].sum())
    app_totals = (
        active_df.groupby('app_name', as_index=False)['duration_seconds']
        .sum()
        .sort_values('duration_seconds', ascending=False)
    )
    if top_n_apps > 0: app_totals = app_totals.head(top_n_apps)

    out = []
    for _, app_row in app_totals.iterrows():
        app = app_row['app_name']
        app_seconds = int(app_row['duration_seconds'])

        title_totals = (
            active_df[active_df['app_name'] == app]
            .groupby('clean_title', as_index=False)['duration_seconds']
            .sum()
            .sort_values('duration_seconds', ascending=False)
        )
        if top_n_titles > 0: title_totals = title_totals.head(top_n_titles)

        top_titles = [
            {
                'title': r['clean_title'],
                'duration_seconds': int(r['duration_seconds'])
            }
            for _, r in title_totals.iterrows()
        ]

        out.append({
            'app': app,
            'duration_seconds': app_seconds,
            'percent_active': round(100 * app_seconds / total_active_seconds, 1),
            'top_titles': top_titles,
        })

    return out


In [ ]:
#| export
def _summary_data(events_df):
    if len(events_df) == 0:
        return {
            'total_duration_seconds': 0,
            'active_duration_seconds': 0,
            'idle_duration_seconds': 0,
            'active_percent': 0.0,
            'idle_percent': 0.0,
            'app_switch_count': 0,
            'distinct_app_count': 0,
            'top_app': None,
            'top_app_duration_seconds': 0,
            'longest_idle_seconds': 0,
        }

    df = events_df.sort_values('start_time').copy()
    total_duration_seconds = int(df['duration_seconds'].sum())

    idle_df = df[df['is_idle'] == True]
    active_df = df[df['is_idle'] == False]

    idle_duration_seconds = int(idle_df['duration_seconds'].sum())
    active_duration_seconds = int(active_df['duration_seconds'].sum())

    active_percent = round(100 * active_duration_seconds / total_duration_seconds, 1) if total_duration_seconds else 0.0
    idle_percent = round(100 * idle_duration_seconds / total_duration_seconds, 1) if total_duration_seconds else 0.0

    distinct_app_count = int(active_df['app_name'].nunique()) if len(active_df) else 0
    app_switch_count = int(active_df['app_name'].ne(active_df['app_name'].shift()).sum() - 1) if len(active_df) else 0

    if len(active_df):
        app_totals = (
            active_df.groupby('app_name', as_index=False)['duration_seconds']
            .sum()
            .sort_values('duration_seconds', ascending=False)
        )
        top_app = app_totals.iloc[0]['app_name']
        top_app_duration_seconds = int(app_totals.iloc[0]['duration_seconds'])
    else:
        top_app = None
        top_app_duration_seconds = 0

    longest_idle_seconds = int(idle_df['duration_seconds'].max()) if len(idle_df) else 0

    return {
        'total_duration_seconds': total_duration_seconds,
        'active_duration_seconds': active_duration_seconds,
        'idle_duration_seconds': idle_duration_seconds,
        'active_percent': active_percent,
        'idle_percent': idle_percent,
        'app_switch_count': app_switch_count,
        'distinct_app_count': distinct_app_count,
        'top_app': top_app,
        'top_app_duration_seconds': top_app_duration_seconds,
        'longest_idle_seconds': longest_idle_seconds,
    }

- `_clean_events` derives `app_name` from `window_title` rather than using the existing event `app` field. Confirm that this is intentional.
- A title without `" - "` becomes the complete `app_name`, while its `clean_title` becomes `"Unknown"`. Titles containing several separators retain only the final two components.
- `_clean_events` assumes every event has non-null, parseable `start_time`, `end_time`, and string `window_title` values. Confirm that the database layer guarantees these conditions.
- `_session_data` assumes both session timestamps are present and parseable. It cannot directly process an open session with `end_time=None`.
- Session duration is calculated from the session timestamps, while summary duration is the sum of event durations. These values can differ when events contain gaps or overlaps.
- `app_switch_count` is calculated after idle rows are removed. Consequently, `App A → Idle → App A` is treated as no application switch.
- The current title breakdown returns title duration but not the requested percent within the application. It also does not return an explicit title count or single top-title field.
- A non-positive `top_n_apps` or `top_n_titles` value means “include all,” not “include none.” Confirm that this matches the intended configuration semantics.
- The configured top-app and top-title limits are captured as default arguments when the module is imported. Runtime changes to those configuration attributes will not change the defaults until the module is reloaded.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()